# COM299 Applied Data Science
## Initial Inspection of the STATS19 Collision Dataset

This notebook inspects the 2025 STATS19 collision dataset for the SB4 Insurance Analytics Team brief.

The purpose is to understand the structure of the collision data, identify circumstantial factors that may later be associated with casualty severity, and inspect the fields needed to link collision records to the casualty and vehicle datasets.

### Load the collision dataset

The 2025 STATS19 collision dataset is loaded from the local `data/raw` folder.

The raw file is kept unchanged so that the inspection and later cleaning steps remain reproducible and can always be traced back to the original source data.

In [1]:
import pandas as pd

collision_path = "../data/raw/dft-road-casualty-statistics-collision-2025.csv"
collisions = pd.read_csv(collision_path)

### Check the size of the dataset

The number of rows and columns is checked first to understand the overall size and structure of the collision table before examining individual variables.

In [2]:
collisions.shape

(101525, 44)

**Result:** The collision dataset contains 101,525 rows and 44 columns.

Each row is expected to represent one recorded collision, with the columns describing collision-level circumstances and conditions.

This table is smaller than the casualty and vehicle tables, which is consistent with a single collision being able to involve multiple vehicles and multiple casualties.

### Inspect the collision dataset columns

The column names are inspected to identify the available collision-level variables, linkage fields, and potential circumstantial factors relevant to the SB4 question.

In [4]:
collisions.columns.tolist()

['collision_index',
 'collision_year',
 'collision_ref_no',
 'location_easting_osgr',
 'location_northing_osgr',
 'longitude',
 'latitude',
 'police_force',
 'collision_severity',
 'number_of_vehicles',
 'number_of_casualties',
 'date',
 'day_of_week',
 'time',
 'local_authority_district',
 'local_authority_ons_district',
 'local_authority_highway',
 'local_authority_highway_current',
 'first_road_class',
 'first_road_number',
 'road_type',
 'speed_limit',
 'junction_detail_historic',
 'junction_detail',
 'junction_control',
 'second_road_class',
 'second_road_number',
 'pedestrian_crossing_human_control_historic',
 'pedestrian_crossing_physical_facilities_historic',
 'pedestrian_crossing',
 'light_conditions',
 'weather_conditions',
 'road_surface_conditions',
 'special_conditions_at_site',
 'carriageway_hazards_historic',
 'carriageway_hazards',
 'urban_or_rural_area',
 'did_police_officer_attend_scene_of_accident',
 'trunk_road_flag',
 'lsoa_of_accident_location',
 'enhanced_severit

**Result:** The collision dataset contains linkage fields, location information, collision severity, road characteristics, environmental conditions, and other circumstantial variables.

The main linkage field is `collision_index`, which should allow the collision table to connect to the casualty and vehicle datasets.

Several fields appear particularly relevant to the SB4 question, including:

- `road_type`
- `speed_limit`
- `junction_detail`
- `junction_control`
- `light_conditions`
- `weather_conditions`
- `road_surface_conditions`
- `special_conditions_at_site`
- `carriageway_hazards`
- `urban_or_rural_area`

These are only candidate variables at this stage. Their coding, completeness, and suitability still need to be inspected before any are selected for analysis.

### Check collision record uniqueness

Before using the collision table in later joins, the collision identifier should be checked for duplicates.

If `collision_index` is unique, this supports the interpretation that each row represents one collision.

In [5]:
collisions.duplicated(subset=["collision_index"]).sum()

np.int64(0)

**Result:** No duplicate `collision_index` values were found.

This supports the interpretation that each row in the collision dataset represents one unique collision.

The `collision_index` field therefore appears suitable as the primary key for the collision table and should be appropriate for linking collision-level circumstances to the casualty and vehicle datasets later.

### Check missing values in collision linkage fields

The main linkage fields should be complete before they are used to connect the collision table to the casualty and vehicle datasets.

A missing-value check is therefore performed on the collision identifiers.

In [6]:
collisions[["collision_index", "collision_ref_no"]].isna().sum()

collision_index     0
collision_ref_no    0
dtype: int64

**Result:** No standard missing values were found in either `collision_index` or `collision_ref_no`.

This means the main collision identifiers are structurally complete and should be reliable for later linkage with the casualty and vehicle datasets.

As with the other STATS19 tables, this does not yet confirm that every identifier matches successfully across tables; that relationship will be validated separately in the join-validation notebook.

### Inspect collision severity

The `collision_severity` field records the overall severity classification of each collision.

Although the SB4 question focuses on casualty severity, inspecting collision severity helps us understand the collision-level structure and how severity is represented in this table.

In [7]:
collisions["collision_severity"].value_counts(dropna=False).sort_index()

collision_severity
1     1453
2    25191
3    74881
Name: count, dtype: int64

**Result:** The `collision_severity` field contains three coded categories:

- `1`: 1,453 collisions
- `2`: 25,191 collisions
- `3`: 74,881 collisions

This shows that most recorded collisions fall into code `3`, with fewer in codes `2` and `1`.

The exact code meanings should be confirmed against the official STATS19/STATS20 documentation before interpretation.

For this project, `collision_severity` is useful contextual information, but it is not the main outcome variable because the SB4 question focuses on casualty severity at the individual casualty level.

### Inspect speed limit values

The `speed_limit` field records the posted speed limit at the collision location.

This may be relevant to SB4 because road speed environment could be associated with casualty severity.

The values are inspected first to identify the available categories and any unusual or sentinel-style entries.

In [8]:
collisions["speed_limit"].value_counts(dropna=False).sort_index()

speed_limit
20    20045
30    49673
40     8886
50     4612
60    12709
70     5600
Name: count, dtype: int64

**Result:** The `speed_limit` field contains six values: `20`, `30`, `40`, `50`, `60`, and `70`.

The most common speed limit is `30 mph`, with 49,673 collisions, followed by `20 mph` and `60 mph`.

No standard missing values or obvious sentinel-style values appear in this field.

Because speed limit is directly interpretable, has good coverage, and is relevant to the collision circumstances, it appears to be a strong candidate for later SB4 analysis.

### Inspect light condition categories

The `light_conditions` field describes the lighting conditions at the time of the collision.

This may be relevant to SB4 because visibility and lighting conditions could be associated with casualty severity.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [9]:
collisions["light_conditions"].value_counts(dropna=False).sort_index()

light_conditions
-1       16
 1    72814
 4    20705
 5      720
 6     5682
 7     1588
Name: count, dtype: int64

**Result:** The `light_conditions` field contains several coded categories.

Most collisions are recorded under code `1`, with smaller groups in codes `4`, `5`, `6`, and `7`. Only 16 records use the sentinel-style value `-1`.

This suggests that the field has very good coverage and could be useful for later SB4 analysis.

The exact meaning of each code should still be confirmed using the official STATS19/STATS20 lookup before interpretation.

### Inspect weather condition categories

The `weather_conditions` field describes the weather at the time of the collision.

This may be relevant to SB4 because adverse weather could be associated with casualty severity.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [10]:
collisions["weather_conditions"].value_counts(dropna=False).sort_index()

weather_conditions
1    83405
2    10230
3      235
4      715
5      880
6       19
7      342
8     3314
9     2385
Name: count, dtype: int64

**Result:** The `weather_conditions` field contains several coded categories.

Most collisions are recorded under code `1`, with smaller numbers across the remaining categories.

No standard missing values or obvious `-1` sentinel-style values appear in this field.

This suggests that `weather_conditions` has good coverage and may be suitable for later SB4 analysis, although the exact meaning of each code should still be confirmed using the official STATS19/STATS20 lookup.

### Inspect road surface condition categories

The `road_surface_conditions` field describes the condition of the road surface at the time of the collision.

This may be relevant to SB4 because wet, icy, or otherwise adverse road conditions could be associated with casualty severity.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [11]:
collisions["road_surface_conditions"].value_counts(dropna=False).sort_index()

road_surface_conditions
-1      634
 1    75679
 2    22312
 3      146
 4     1261
 5      151
 9     1342
Name: count, dtype: int64

**Result:** The `road_surface_conditions` field contains several coded categories.

Most collisions are recorded under code `1`, followed by code `2`, with much smaller numbers across the remaining categories.

The value `-1` appears in 634 records and code `9` also appears in 1,342 records. These should be checked against the official STATS19/STATS20 lookup before interpretation.

Overall, the field appears to have good coverage and may be a useful circumstantial variable for later SB4 analysis.

### Inspect road type categories

The `road_type` field describes the type of road on which the collision occurred.

This may be relevant to SB4 because different road environments could be associated with different casualty severity outcomes.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [12]:
collisions["road_type"].value_counts(dropna=False).sort_index()

road_type
1     6795
2     2398
3    14610
6    74150
7     1619
9     1953
Name: count, dtype: int64

**Result:** The `road_type` field contains six coded categories.

Most collisions are recorded under code `6`, with the remaining records spread across codes `1`, `2`, `3`, `7`, and `9`.

There are no standard missing values or obvious `-1` sentinel-style values in this field.

Because road type is a direct description of the collision environment and appears to have good coverage, it may be a useful candidate for later SB4 analysis.

The exact meaning of each code should still be confirmed using the official STATS19/STATS20 lookup before interpretation.

### Inspect urban or rural area categories

The `urban_or_rural_area` field identifies the broad environment in which the collision occurred.

This may be relevant to SB4 because casualty severity could differ between urban and rural road environments.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [13]:
collisions["urban_or_rural_area"].value_counts(dropna=False).sort_index()

urban_or_rural_area
1    67148
2    34373
3        4
Name: count, dtype: int64

**Result:** The `urban_or_rural_area` field contains three coded categories.

Most collisions are recorded under code `1`, with a smaller number under code `2`. Only four records use code `3`.

This suggests the field is highly complete and could provide a simple way to compare broad road environments in the later SB4 analysis.

The exact meaning of codes `1`, `2`, and `3` should still be confirmed using the official STATS19/STATS20 lookup before interpretation.

### Inspect junction detail categories

The `junction_detail` field describes the type of junction associated with the collision.

This may be relevant to SB4 because different junction layouts could be associated with different casualty severity outcomes.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [14]:
collisions["junction_detail"].value_counts(dropna=False).sort_index()

junction_detail
-1      6701
 0     45333
 13    28198
 16     9456
 17     1504
 18     2144
 19     6762
 99     1427
Name: count, dtype: int64

**Result:** The `junction_detail` field contains several coded categories.

The largest groups are code `0` and code `13`, with smaller numbers across codes `16`, `17`, `18`, and `19`.

The value `-1` appears in 6,701 records and code `99` appears in 1,427 records. These are likely special or unknown-style categories and should be confirmed using the official STATS19/STATS20 lookup before interpretation.

Because the field contains several categories and some special values, it may still be useful for SB4, but it will require more careful handling than simpler variables such as `speed_limit` or `urban_or_rural_area`.

### Inspect junction control categories

The `junction_control` field describes how traffic is controlled at the junction.

This may be relevant to SB4 because different traffic-control arrangements could be associated with casualty severity.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [15]:
collisions["junction_control"].value_counts(dropna=False).sort_index()

junction_control
-1    44487
 1      381
 2    10580
 3      649
 4    41902
 9     3526
Name: count, dtype: int64

**Result:** The `junction_control` field contains several coded categories.

The two largest groups are code `-1` with 44,487 records and code `4` with 41,902 records. Code `2` is also relatively common, while the remaining categories are much smaller.

The high number of `-1` values suggests that junction control information is unavailable, not applicable, or otherwise specially coded for a substantial proportion of collisions.

Because of this, `junction_control` may be less suitable for later SB4 analysis than better-covered variables such as `speed_limit`, `light_conditions`, or `urban_or_rural_area`.

The exact meaning of each code should still be confirmed using the official STATS19/STATS20 lookup.

### Check missing values across the collision dataset

Missing data can affect whether collision-level variables are suitable for later analysis.

A first-pass missing-value check is used to identify any standard `NaN` values across the collision dataset before cleaning decisions are made.

In [16]:
collisions.isna().sum()

collision_index                                     0
collision_year                                      0
collision_ref_no                                    0
location_easting_osgr                               2
location_northing_osgr                              2
longitude                                           2
latitude                                            2
police_force                                        0
collision_severity                                  0
number_of_vehicles                                  0
number_of_casualties                                0
date                                                0
day_of_week                                         0
time                                                0
local_authority_district                            0
local_authority_ons_district                        0
local_authority_highway                             0
local_authority_highway_current                     0
first_road_class            

**Result:** The collision dataset contains very few standard missing values.

Only the location fields contain `NaN` values:

- `location_easting_osgr`: 2 missing
- `location_northing_osgr`: 2 missing
- `longitude`: 2 missing
- `latitude`: 2 missing

All other columns contain no standard `NaN` values.

This means the collision dataset is structurally very complete, although sentinel-style and coded unknown values are still present in several categorical fields.

The small amount of missing location data is unlikely to affect the main SB4 analysis unless geographic variables are used later.

### Inspect the number of casualties per collision

The `number_of_casualties` field records how many casualties were associated with each collision.

Inspecting this field helps confirm the one-to-many relationship expected between the collision table and the casualty table.

In [17]:
collisions["number_of_casualties"].value_counts().sort_index()

number_of_casualties
1      83317
2      13126
3       3301
4       1178
5        389
6        113
7         46
8         17
9          9
10         9
11         3
12         6
13         1
15         1
17         1
18         2
20         2
22         1
25         1
32         1
142        1
Name: count, dtype: int64

**Result:** Most collisions involve a single casualty, with 83,317 collisions recording one casualty.

However, some collisions involve multiple casualties, including a small number of unusually large cases. The maximum recorded value is 142 casualties in a single collision.

This confirms that the relationship between the collision and casualty tables is one-to-many: one collision can be linked to several casualty records.

This further supports using the casualty as the likely unit of analysis for the SB4 question, because casualty severity is recorded separately for each individual casualty.